In [75]:
import numpy as np
import pandas as pd
import requests
import os
from multiprocessing import Pool, cpu_count

from ml.preprocessor import preprocess_data
from ml.pipeline import get_whitelist, search_whitelist, run_model


TIMEOUT = 4
DATA_FILE = "../data/legitphish_dataset.csv"
INVALID_FILE = "../outputs/legitphish_dataset_invalid.csv"
TIMED_OUT_FILE = "../outputs/legitphish_dataset_timed_out.csv"
VALID_FILE = "../outputs/legitphish_dataset_valid.csv"

In [76]:
dataset_df = pd.DataFrame()
try:
    dataset_df = pd.read_csv(DATA_FILE)
except pd.errors.EmptyDataError:
    pass

dataset_df = dataset_df[["URL", "ClassLabel"]]

invalid_urls = pd.DataFrame(columns=["url", "status_code"])
try:
    invalid_urls = pd.read_csv(INVALID_FILE)
except pd.errors.EmptyDataError:
    pass

timed_out_urls = pd.DataFrame(columns=["url", "error"])
try:
    timed_out_urls = pd.read_csv(TIMED_OUT_FILE)
except pd.errors.EmptyDataError:
    pass

valid_urls = pd.DataFrame(columns=["url", "is_legit"])
try:
    valid_urls = pd.read_csv(VALID_FILE)
except pd.errors.EmptyDataError:
    pass


In [77]:
dataset_dict = dict(zip(dataset_df["URL"], dataset_df["ClassLabel"]))
for idx, row in invalid_urls.iterrows():
    if row["status_code"] >= 200 and row["status_code"] < 400:
        url = row["url"]
        corresponding_class_label = dataset_dict[url]
        valid_urls.loc[len(valid_urls)] = [url, corresponding_class_label]

valid_urls.to_csv(VALID_FILE, index=False)


In [78]:
dataset_dict = dict(zip(dataset_df["URL"], dataset_df["ClassLabel"]))

missing_scheme = []
for idx, row in timed_out_urls.iterrows():
    if "No scheme supplied" in row["error"]:
        url = row["url"]
        timed_out_urls.drop(idx, inplace=True)
        missing_scheme.append(url)

In [79]:
def test_url(url, label):
    try:
        r = requests.head(url, allow_redirects=True, timeout=TIMEOUT)
        if r.status_code >= 200 and r.status_code < 400:
            return ("valid", url, label)
        else:
            return ("invalid", url, r.status_code)
    except Exception as e:
        return ("timeout", url, str(e))


def test_url_unpack(args):
    return test_url(*args)

In [80]:
tasks = [("http://" + url, dataset_dict[url]) for url in missing_scheme if url in dataset_dict]
with Pool(processes=cpu_count()) as pool:
    for i, result in enumerate(pool.imap(test_url_unpack, tasks)):
        result_type, url, value = result

        if result_type == "invalid":
            invalid_urls.loc[len(invalid_urls)] = [url, value]
        elif result_type == "timeout":
            timed_out_urls.loc[len(timed_out_urls)] = [url, value]
        elif result_type == "valid":
            valid_urls.loc[len(valid_urls)] = [url, value]

        if (i + 1) % 100 == 0:
            print(f"Processed {i + 1}")

        if (i + 1) % 1000 == 0:
            invalid_urls.to_csv(INVALID_FILE, index=False)
            timed_out_urls.to_csv(TIMED_OUT_FILE, index=False)
            valid_urls.to_csv(VALID_FILE, index=False)

    invalid_urls.to_csv(INVALID_FILE, index=False)
    timed_out_urls.to_csv(TIMED_OUT_FILE, index=False)
    valid_urls.to_csv(VALID_FILE, index=False)

Processed 100
Processed 200
Processed 300
Processed 400
Processed 500
Processed 600
Processed 700
Processed 800
Processed 900
Processed 1000
Processed 1100
Processed 1200
